In [7]:
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import BaggingRegressor, BaggingClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, r2_score

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
DATA = Path("datasets")

In [3]:
df = pd.read_csv(DATA / 'bank_data_processed_2.csv')

In [4]:
df

,Unnamed: 0,Age,Income,Family,Education,Mortgage,Securities Account,CD Account,Online,CreditCard
0,0,34,180,1,3,0,0,0,0,0
1,1,38,130,4,3,134,0,0,0,0
2,2,46,193,2,3,0,0,0,0,0
3,3,38,119,1,2,0,0,1,1,1
4,4,42,141,3,3,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...
475,475,38,129,3,3,0,0,1,1,1
476,476,43,121,1,2,0,0,1,1,1
477,477,28,112,2,2,0,0,0,1,0
478,478,46,122,3,3,0,0,1,1,1


In [5]:
X = df.drop(columns="CreditCard")
y= df["CreditCard"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
bag_reg = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=400,
    bootstrap=False, #False to pasting, True to bagging
    max_samples = 1.0, # 80% danych jest brane do modelu
    n_jobs=-1,
    random_state=42
)

bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)

accuracy_score(y_test, y_pred)

0.7708333333333334

In [12]:
bag_reg = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=400,
    bootstrap=True, #False to pasting, True to bagging
    max_samples = 0.7, # 80% danych jest brane do modelu
    n_jobs=-1,
    random_state=42,
    oob_score=True
)

bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)

accuracy_score(y_test, y_pred)

0.8125

In [14]:
bag_reg = BaggingClassifier(
    DecisionTreeClassifier(random_state=42),
    n_estimators=400,
    bootstrap=True, #False to pasting, True to bagging
    max_samples = 1.0, # 80% danych jest brane do modelu
    n_jobs=-1,
    random_state=42,
    oob_score=True,
    bootstrap_features=True, 
)

bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)

accuracy_score(y_test, y_pred)

0.84375

In [17]:
from scipy.stats import randint, uniform

from sklearn.model_selection import RandomizedSearchCV, KFold,GridSearchCV

In [20]:
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

param_distributions = {
    "n_estimators": randint(150, 500),
    "max_samples": uniform(0.5, 0.4),
    "estimator__max_depth": randint(5, 20),
    "estimator__min_samples_leaf": randint(1, 15),
    "estimator__min_samples_split": randint(2, 30)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)


random_search = RandomizedSearchCV(
    estimator=bag_clf,
    param_distributions=param_distributions,
    n_iter=40,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42
)


random_search.fit(X_train, y_train)

print("Best CV accuracy: ", random_search.best_score_)
print("Best params: ", random_search.best_params_)

best_model = random_search.best_estimator_

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV accuracy:  0.8099453178400549
Best params:  {'estimator__max_depth': 8, 'estimator__min_samples_leaf': 14, 'estimator__min_samples_split': 6, 'max_samples': np.float64(0.59971689165955), 'n_estimators': 386}


In [21]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)


In [24]:
accuracy_score(y_test, y_pred)

0.90625